In [4]:
"""
parte 1:
Receber entradas do usuário em loop (modo conversacional) = feito
- Processar texto digitado (tratamento básico) = feito
- Identificar padrões ou intenções na mensagem = feito (agora via Machine Learning)
- Gerar respostas apropriadas = feito
- Encerrar quando o usuário desejar = feito

parte 2:
Utilizar estruturas de decisão (if/elif/else) = feito
Utilizar laços de repetição = feito
Trabalhar com listas ou estruturas similares = feito
"""

#Bibliotecas
#===============================================================================
import unicodedata
import time
import random
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
#===============================================================================

# Funções
#==============================================================================
def normalizar(texto):

    texto = texto.lower().strip()

    texto = texto.replace(".", " ")
    texto = texto.replace(",", " ")
    texto = texto.replace(";", " ")
    texto = texto.replace(":", " ")
    texto = texto.replace("!", " ")
    texto = texto.replace("?", " ")
    texto = texto.replace("(", " ")
    texto = texto.replace(")", " ")
    texto = texto.replace("[", " ")
    texto = texto.replace("]", " ")
    texto = texto.replace("{", " ")
    texto = texto.replace("}", " ")
    texto = texto.replace("/", " ")
    texto = texto.replace("\\", " ")
    texto = texto.replace("-", " ")
    texto = texto.replace("_", " ")
    texto = texto.replace('"', "")
    texto = texto.replace("'", "")
    texto = texto.replace("@", "")
    texto = texto.replace("#", "")
    texto = texto.replace("%", "")

    texto = " ".join(texto.split())

    texto = unicodedata.normalize("NFD", texto)

    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

    return texto


def remover_palavras_ignoradas(texto):
    palavras = texto.split()

    palavras_filtradas = [
        palavra
        for palavra in palavras
        if palavra not in palavras_ignorar
    ]

    if not palavras_filtradas:
        return texto

    return " ".join(palavras_filtradas)


def preparar_texto(texto):
    texto = normalizar(texto)
    texto = remover_palavras_ignoradas(texto)
    return texto


def prever_intencao(texto):
    texto_preparado = preparar_texto(texto)
    vetor = vectorizer.transform([texto_preparado])

    if vetor.sum() == 0:
        return "desconhecida"

    return modelo.predict(vetor)[0]


def gerar_resposta(intencao):
    if intencao == "saudacao":
        resposta = random.choice(respostas_saudacao)

    elif intencao == "ajuda":
        resposta = random.choice(respostas_ajuda)

    elif intencao == "informacao":
        resposta = random.choice(respostas_informacao)

    elif intencao == "produto":
        resposta = random.choice(respostas_produto)

    elif intencao == "preco":
        resposta = random.choice(respostas_preco)

    elif intencao == "pagamento":
        resposta = random.choice(respostas_pagamento)

    elif intencao == "entrega":
        resposta = random.choice(respostas_entrega)

    elif intencao == "despedida":
        resposta = random.choice(respostas_despedida)

    else:
        resposta = random.choice(respostas_desconhecida)

    return resposta


def encontrar_produto(texto):
    texto = normalizar(texto)

    for nome in catalogo:
        if nome in texto:
            return nome

    for apelido, nome in apelidos.items():
        if apelido in texto:
            return nome

    return None


def listar_produtos():
    nomes = list(catalogo)
    return ", ".join(nomes[:-1]) + " e " + nomes[-1]


def encontrar_modelo(produto, texto):
    modelos = catalogo[produto].get("modelos", {})

    for palavra, resposta in modelos.items():
        if palavra in texto:
            return resposta

    return None


def gerar_resposta_contextual(intencao, entrada, contexto):
    tipo, ultimo_produto = contexto
    texto = normalizar(entrada)
    produto = encontrar_produto(entrada)

    base = produto or ultimo_produto

    if base and intencao != "despedida":

        resposta_modelo = encontrar_modelo(base, texto)
        if resposta_modelo:
            return f"{resposta_modelo} Posso ajudar em mais alguma coisa?", (None, base)

        if produto:
            dados = catalogo[produto]

            if "modelos" in dados:
                resposta = dados["descricao"] + " " + " ".join(dados["modelos"].values())
            else:
                resposta = dados["descricao"] + " " + dados["preco"]

            return f"{resposta} Posso ajudar em mais alguma coisa?", (None, produto)

    resposta = gerar_resposta(intencao)

    if intencao == "produto":
        resposta = f"{resposta} Hoje temos: {listar_produtos()}. Qual voce quer ver?"

    return resposta, (None, None)
#==============================================================================


#Dicionário
#==============================================================================

respostas_saudacao = [
    "Ola! Seja bem-vindo a loja do Fluminense. Como posso te ajudar?",
    "Oi! Tudo bem? O que voce esta procurando na nossa loja?",
    "Fala, tricolor! Como posso ajudar voce hoje?",
    "Ola! Seja muito bem-vindo. Em que posso te ajudar?",
    "Oi! Que bom ter voce por aqui. O que voce gostaria de encontrar?",
    "Bom dia! Como posso ajudar voce na loja do Fluminense?",
    "Boa tarde! O que voce esta procurando hoje?",
    "Boa noite! Seja bem-vindo a nossa loja.",
    "Ola, torcedor! Pode me dizer o que voce esta procurando?",
    "E ai! Tudo certo? Vamos encontrar o produto que voce procura?"
]


respostas_ajuda = [
    "Claro! Me conta o que voce esta procurando que eu te ajudo.",
    "Pode deixar, vou tentar te ajudar da melhor forma possivel.",
    "Sem problemas! Me explica sua duvida.",
    "Claro que sim. O que voce precisa saber?",
    "Estou aqui para ajudar. Pode me explicar o que aconteceu?",
    "Vamos resolver isso juntos. O que voce esta procurando?",
    "Pode falar. Vou tentar encontrar a melhor orientacao para voce.",
    "Me explica melhor sua duvida para que eu possa te ajudar.",
    "Claro! Se voce me disser o que precisa, consigo te orientar.",
    "Fique tranquilo. Pode me contar o que esta acontecendo."
]


respostas_informacao = [
    "Claro! Quer saber o horario de funcionamento, o endereco ou o contato da loja?",
    "Posso te passar informacoes gerais sobre a nossa loja, como horario e localizacao.",
    "Me diga qual informacao da loja voce esta procurando.",
    "Sem problemas. Voce quer saber sobre horario, endereco ou atendimento?",
    "Pode perguntar. Vou tentar te passar a informacao sobre a loja que voce precisa.",
    "Claro! Quer saber sobre o funcionamento, horario ou localizacao da loja?",
    "Posso ajudar com telefone, endereco e formas de contato da loja.",
    "Pode me dizer exatamente o que voce gostaria de saber sobre a loja?",
    "Vamos la. Qual informacao sobre a loja voce esta procurando?",
    "Claro! Quer saber se temos loja fisica ou como entrar em contato?"
]


respostas_despedida = [
    "Foi um prazer te atender. Volte sempre!",
    "Obrigado pelo contato. Esperamos voce novamente!",
    "Foi muito bom ajudar voce. Ate a proxima!",
    "Obrigado pela visita. Tenha um excelente dia!",
    "Pode contar conosco quando precisar. Ate mais!",
    "Espero ter conseguido ajudar. Ate a proxima!",
    "Obrigado pelo atendimento. Volte sempre a nossa loja!",
    "Foi um prazer conversar com voce. Tenha um otimo dia!",
    "Agradecemos pelo contato. Nos vemos em breve!",
    "Obrigado por escolher nossa loja. Ate mais!"
]


respostas_desconhecida = [
    "Desculpa, nao consegui entender sua mensagem. Pode explicar de outra forma?",
    "Acho que nao entendi sua duvida. Pode explicar melhor?",
    "Nao consegui identificar o que voce esta procurando.",
    "Desculpa, pode repetir a pergunta para mim?",
    "Quero te ajudar, mas preciso entender melhor o que voce precisa.",
    "Nao tenho certeza se entendi corretamente. Pode reformular sua pergunta?",
    "Vamos tentar novamente? Me diga com outras palavras o que voce precisa.",
    "Ainda nao consegui entender sua solicitacao. Pode dar mais detalhes?",
    "Desculpa pela confusao. Pode explicar melhor o que voce procura?",
    "Nao consegui encontrar uma resposta para isso. Pode me explicar melhor?"
]


respostas_pagamento = [
    "O pagamento e feito somente na loja, na hora da compra.",
    "Aqui o pagamento e presencial: voce paga direto na loja.",
    "Nao temos pagamento online. Tudo e resolvido pessoalmente, na loja.",
    "Para pagar, e so vir ate a loja. O pagamento e feito la mesmo.",
    "Nosso pagamento e apenas na loja, no momento em que voce escolhe o produto.",
    "Voce faz o pagamento presencialmente, direto no caixa da loja.",
    "As compras sao pagas na loja. Nao fazemos cobranca a distancia.",
    "Pode vir tranquilo: voce escolhe seu produto e paga na propria loja.",
    "Aqui o pagamento acontece so na loja, na hora de retirar o seu produto.",
    "O pagamento e feito na loja. Nao trabalhamos com pagamento fora dela."
]


respostas_produto = [
    "Claro! Temos diversos produtos do Fluminense.",
    "Com certeza! Temos varias opcoes tricolores.",
    "Posso te ajudar com isso! Veja o que temos.",
    "Boa escolha! Temos produtos para todo tricolor.",
    "Temos diferentes tipos de produtos.",
    "Claro! Temos opcoes para voce ou para presente."
]


respostas_preco = [
    "O valor depende do modelo e do tamanho. Qual camisa ou produto voce quer consultar?",
    "Os precos variam de produto para produto. Me diga qual item voce procura que eu te oriento.",
    "Para te passar o valor certo, preciso saber o modelo. Camisa titular, reserva ou outro produto?",
    "O preco muda conforme o modelo. Qual deles te interessou?",
    "Voce encontra o valor de cada item na pagina do produto. Qual deles voce quer ver?"
]


respostas_entrega = [
    "Nao fazemos entrega. Todas as compras sao feitas e retiradas na loja.",
    "Aqui nao temos entrega: voce escolhe e leva o seu produto direto na loja.",
    "No momento nao entregamos pedidos. O atendimento e somente na loja.",
    "Nao trabalhamos com envio. Para comprar, e so vir ate a nossa loja.",
    "A gente nao faz entrega, mas voce pode vir buscar seu produto na loja.",
    "Nossas vendas sao presenciais, entao nao ha entrega nem frete.",
    "Nao temos frete nem rastreamento, porque tudo e resolvido na loja.",
    "Entrega nao e com a gente: voce retira o seu produto pessoalmente na loja.",
    "Nao enviamos produtos. Venha ate a loja e leve o seu na hora.",
    "Aqui o atendimento e so na loja: nao fazemos entrega, e voce leva o produto na hora."
]

catalogo = {
    "camisa": {
        "descricao": "Temos camisa titular, reserva, infantil, feminina e de goleiro.",
        "modelos": {
            "titular": "Titular: R$ 249,90.",
            "reserva": "Reserva: R$ 229,90.",
            "infantil": "Infantil: R$ 149,90.",
            "feminin": "Feminina: R$ 199,90.",
            "masculin": "Masculina: a partir de R$ 199,90.",
            "goleiro": "Goleiro: R$ 259,90."
        }
    },
    "bone": {
        "descricao": "Temos bones nas cores do Fluminense.",
        "preco": "Os bones custam a partir de R$ 59,90 (valor de exemplo)."
    },
    "bandeira": {
        "descricao": "Temos bandeiras do Fluminense em varios tamanhos.",
        "preco": "As bandeiras custam a partir de R$ 49,90 (valor de exemplo)."
    },
    "cachecol": {
        "descricao": "Temos cachecois tricolores.",
        "preco": "Os cachecois custam a partir de R$ 39,90 (valor de exemplo)."
    },
    "short": {
        "descricao": "Temos shorts nas cores do Fluminense, masculinos e femininos.",
        "modelos": {
            "masculin": "Masculino: R$ 89,90.",
            "feminin": "Feminino: R$ 79,90."
        }
    },
    "bola": {
        "descricao": "Temos bolas oficiais e de treino do Fluminense.",
        "preco": "As bolas custam a partir de R$ 89,90 (valor de exemplo)."
    },
    "chuteira": {
        "descricao": "Temos chuteiras de campo, society e futsal.",
        "preco": "As chuteiras custam a partir de R$ 199,90 (valor de exemplo)."
    }
}

apelidos = {
    "bermuda": "short",
    "calcao": "short",
    "pelota": "bola",
    "chuteiras": "chuteira"
}

palavras_ignorar = {
    "a", "o", "as", "os", "um", "uma", "uns", "umas",
    "de", "do", "da", "dos", "das",
    "em", "no", "na", "nos", "nas",
    "por", "para", "com", "sem",
    "e", "ou", "se",
    "ao", "aos",
    "meu", "minha", "meus", "minhas",
    "seu", "sua", "seus", "suas",
    "eu", "me"
}
#==============================================================================


#Dataset e Modelo de Machine Learning
#==============================================================================

exemplos = {
    "saudacao": [
        "ola",
        "oi",
        "bom dia",
        "boa tarde",
        "boa noite",
        "oi tudo bem",
        "fala tricolor",
        "e ai",
        "e ai tudo certo",
        "ola loja do fluminense",
        "oi loja tricolor",
        "sou torcedor do fluminense",
        "ola sou tricolor",
        "bom dia pessoal",
        "boa tarde pessoal",
        "boa noite pessoal",
        "como voces estao",
        "tem alguem para me atender",
        "quero falar com um atendente",
        "vim conhecer a loja"
    ],

    "ajuda": [
        "ajuda",
        "help",
        "preciso de ajuda",
        "pode me ajudar",
        "me ajuda por favor",
        "estou com uma duvida",
        "pode me orientar",
        "preciso de uma orientacao",
        "me ajuda a escolher",
        "nao consigo encontrar o que procuro",
        "preciso de ajuda para finalizar",
        "pode me explicar melhor",
        "tenho uma duvida",
        "nao sei o que fazer",
        "voce consegue me ajudar",
        "preciso de ajuda com um problema",
        "pode me ajudar com uma duvida",
        "estou tendo dificuldade",
        "nao estou conseguindo comprar",
        "pode me dar uma ajuda",
        "tem como me orientar",
        "quero ajuda para escolher"
    ],

    "informacao": [
        "quero uma informacao",
        "preciso de uma informacao",
        "quero saber mais sobre a loja",
        "onde fica a loja",
        "qual o endereco da loja",
        "qual o horario da loja",
        "qual o horario de funcionamento",
        "a loja funciona no domingo",
        "a loja abre no sabado",
        "onde posso encontrar a loja",
        "quero informacoes sobre a loja",
        "como funciona a loja",
        "quero conhecer a loja",
        "voces tem novidades",
        "quais sao as novidades da loja",
        "como funciona o atendimento",
        "posso ir buscar na loja",
        "tem loja fisica",
        "qual o telefone da loja",
        "como entro em contato com a loja",
        "o que voce faz",
        "quem e voce",
        "quem e voce pedro",
        "para que voce serve",
        "voce e um atendente virtual",
        "voce e um robo"
    ],

    "despedida": [
        "sair",
        "encerrar",
        "adeus",
        "tchau",
        "ate mais",
        "ate logo",
        "ate a proxima",
        "obrigado",
        "valeu",
        "muito obrigado pela ajuda",
        "obrigado pelo atendimento",
        "ja consegui o que precisava",
        "por hoje e so",
        "nao preciso de mais nada",
        "pode encerrar",
        "vou embora",
        "foi isso obrigado",
        "obrigado pela atencao",
        "valeu pela ajuda",
        "tenha um bom dia",
        "tenha uma boa noite",
        "ate mais tricolor",
        "foi um prazer"
    ],

    "desconhecida": [
        "qual a capital da franca",
        "me conte uma piada",
        "qual o melhor celular",
        "vai chover hoje",
        "quanto e dois mais dois",
        "me fale sobre filmes",
        "qual seu nome",
        "voce gosta de futebol",
        "me conte uma historia",
        "qual a melhor comida",
        "quem descobriu o brasil",
        "qual o maior planeta",
        "me recomende uma musica",
        "qual a sua idade",
        "voce sabe programar",
        "me fale sobre tecnologia",
        "qual o melhor computador",
        "me conte uma curiosidade",
        "qual o melhor jogo",
        "voce tem sentimentos"
    ],

    "preco": [
        "quanto e a camisa",
        "quanto custa a camisa",
        "quanto custa",
        "qual o preco da camisa",
        "qual o preco",
        "qual o valor da camisa",
        "quanto esta a camisa",
        "quanto sai a camisa",
        "preco da camisa titular",
        "valor da camisa reserva",
        "quanto custa o bone",
        "quanto e o produto",
        "qual o preco do produto",
        "esta quanto",
        "me passa o preco",
        "tem desconto na camisa",
        "tem promocao",
        "tem cupom",
        "quanto e",
        "quanto",
        "quanto sai",
        "quanto fica"
    ],

    "pagamento": [
        "quero saber como pagar",
        "quero saber como pago",
        "quero pagar",
        "como pagar",
        "como pago",
        "como pago no pix",
        "como pago com cartao",
        "como faco para pagar",
        "e pagamento",
        "pagamento",
        "forma de pagamento",
        "pix",
        "quais formas de pagamento",
        "como posso pagar",
        "posso pagar com cartao",
        "aceita cartao de credito",
        "aceita cartao de debito",
        "posso pagar com pix",
        "aceita pix",
        "posso parcelar a compra",
        "quantas vezes posso parcelar",
        "quais sao as formas de pagamento",
        "tem pagamento por pix",
        "posso pagar em dinheiro",
        "aceita dinheiro",
        "tem parcelamento",
        "qual o numero de parcelas",
        "posso dividir minha compra",
        "o pagamento pode ser feito no cartao",
        "quais cartoes voces aceitam",
        "tem pagamento online",
        "como e o pagamento"
    ],

    "produto": [
        "quero comprar uma camisa",
        "tem camisa do fluminense",
        "tem camisa oficial",
        "quais camisas voces vendem",
        "tem camisa titular",
        "tem camisa reserva",
        "tem camisa de goleiro",
        "tem camisa infantil",
        "tem camisa feminina",
        "tem camisa masculina",
        "vende bone do fluminense",
        "vende bandeira do fluminense",
        "vende cachecol do fluminense",
        "vende acessorios do fluminense",
        "tem produtos para presente",
        "quero comprar um produto tricolor",
        "quero ver as camisas disponiveis",
        "quero comprar uma camisa infantil",
        "quero comprar um bone",
        "quero ver os produtos tricolores",
        "quais produtos voces tem",
        "o que voces vendem",
        "tem short",
        "tem short do fluminense",
        "quero comprar short",
        "tem bermuda",
        "vende short",
        "tem bola",
        "vende bola do fluminense",
        "quero comprar uma bola",
        "tem bola oficial",
        "tem chuteira",
        "vende chuteira",
        "quero comprar chuteira",
        "tem chuteira do fluminense",
        "quero ver as chuteiras"
    ],

    "entrega": [
        "entrega",
        "e entrega",
        "frete",
        "voces entregam",
        "entregam em casa",
        "tem entrega em casa",
        "quero saber sobre entrega",
        "voces fazem entrega",
        "qual o prazo de entrega",
        "quanto tempo demora para chegar",
        "qual o valor do frete",
        "quanto custa a entrega",
        "voces entregam em todo o brasil",
        "como e feita a entrega",
        "quando meu pedido vai chegar",
        "posso acompanhar minha entrega",
        "onde esta meu pedido",
        "qual o prazo para receber minha compra",
        "quanto tempo leva o envio",
        "voces entregam em outros estados",
        "tem entrega para o rio de janeiro",
        "como acompanho meu pedido",
        "onde vejo o rastreamento",
        "meu pedido ainda nao chegou",
        "quando minha encomenda chega",
        "como e calculado o frete",
        "tem entrega expressa",
        "quanto custa o frete",
        "quanto e o frete",
        "quanto custa o envio",
        "quanto fica o frete"
    ]
}

textos = []
intencoes = []

for intencao_nome, frases in exemplos.items():
    for frase in frases:
        textos.append(frase)
        intencoes.append(intencao_nome)

df = pd.DataFrame({"texto": textos, "intencao": intencoes})

df["texto"] = df["texto"].apply(preparar_texto)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df["texto"])

modelo = MultinomialNB()
modelo.fit(X, df["intencao"])
#==============================================================================


#Codigo Principal
#===============================================================================

ct = "Pedro"
us = "Usuário Tricolor"

print(f"\n...: Olá! Eu me chamo {ct} e eu estarei te atendendo hoje.\n")

time.sleep(1)

print(f"{ct}: Qual é o seu nome?\n")

nm = input(f"{us}: ").strip() or "Torcedor"

print(f"\n{ct}: Prazer, {nm}! Como posso ajudar você?\n")

contexto = (None, None)

while True:

    entrada = input(f"{nm}: ")

    intencao = prever_intencao(entrada)
    resposta, contexto = gerar_resposta_contextual(intencao, entrada, contexto)

    time.sleep(1)

    print(f"\n{ct}: {resposta}\n")

    if intencao == "despedida":
        break


...: Olá! Eu me chamo Pedro e eu estarei te atendendo hoje.

Pedro: Qual é o seu nome?


Pedro: Prazer, david! Como posso ajudar você?


Pedro: A gente nao faz entrega, mas voce pode vir buscar seu produto na loja.



KeyboardInterrupt: Interrupted by user